<img src='https://tools.actris-ares.eu/wp-content/uploads/2025/07/Actris-150x150.png' align='right' style='max-height: 90px;'>

# Load and plot ACTRIS NRT In-Situ Data

This notebook demonstrates how ACTRIS Near Real-Time (NRT) In-Situ data can be searched, accessed, and interactively plotted for a user-defined time interval. It is intended for interactive exploration of ACTRIS NRT datasets. The users are encouraged to browse stations, inspect variables, and experiment with different time intervals and plot types.

#### Helper modules

The notebook uses helper scripts located in `nrt_src/`:

* `nrt_config.py` — ACTRIS-style instrument and variable mapping for NRT search
* `nrt_utils.py` — THREDDS indexing, filename filtering, time slicing, and dataset utilities
* `nrt_ui.py` — interactive ACTRIS-style widget interface

And shared Plotly plotting functions used for both NRT and ACTRIS In-Situ notebooks from:

* `actris_plotting_utilities.py`

Normally, users only need to run this notebook and do not need to modify the helper scripts.

#### Notebook workflow

1. Set the options in the **Configuration** cell (nothing else needs editing)
2. Import libraries and helper modules
3. Access ACTRIS NRT datasets from THREDDS catalogs
4. Build and de-duplicate the combined NRT file index
5. Select instrument, variable, facility, file, and time interval
6. Interactively visualize the selected dataset using Plotly plots

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

## 0. Configuration

All the knobs for this notebook live in the cell below, so you don't have to hunt through the code:

* `INSTALL_MISSING` — install missing Python packages automatically (only the ones actually missing, so
  the stable native widgets of the D4Science environment are left untouched)
* `SHOW_TABLES` — print the table overview of files and instruments when building the index
* `DEV_RELOAD` — force-reload the helper modules from `nrt_src/`; only needed while editing them
* `THREDDS_TIMEOUT_NOTE` — the catalogs are crawled over the network, so the access cell can take a
  while on a slow connection

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "ACTRIS Aerosol In-Situ NRT"
NOTEBOOK_VERSION = "2.3"
NOTEBOOK_DATE = "2026-10-25"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

# Install missing packages automatically (set to False if the environment is managed centrally)
INSTALL_MISSING = True

# Print the table overview of files and instruments when building the NRT index
SHOW_TABLES = False

# Reload the helper modules from nrt_src/ on every run.
# Keep this False unless you are actively editing those files: reloading re-injects the Plotly
# JavaScript bundle into the notebook outputs (~5 MB each time) and slows the notebook down.
DEV_RELOAD = False

# THREDDS catalog endpoints
THREDDS_ACTRIS_NRT = "https://thredds.nilu.no/thredds/catalog/actris_nrt/catalog.xml"
THREDDS_EBAS = "https://thredds.nilu.no/thredds/catalog/ebas/catalog.xml"



## 1. Load necessary libraries

#### Dependencies

Only the packages that are actually missing get installed, and the install is skipped entirely on
later runs — so re-running the notebook doesn't spend time reinstalling what is already there.

In [ ]:
import os
import importlib.util
import subprocess
import sys

# module name -> pip package name
REQUIRED = {
    "threddsclient": "threddsclient",
    "plotly": "plotly",
    "narwhals": "narwhals",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if not missing:
    print("--> All required packages are already installed.")
elif not INSTALL_MISSING:
    print(f"--> Missing packages: {', '.join(missing)} (INSTALL_MISSING is False, install them manually)")
else:
    print(f"--> Installing missing packages: {', '.join(missing)}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        capture_output=True, text=True,
        # pip reads os.getcwd() at startup; running it from the home directory keeps it working
        # even if the folder the kernel started in has since been replaced.
        cwd=os.path.expanduser("~"),
    )
    if result.returncode == 0:
        print("--> Packages installed successfully!")
    else:
        print("--> Installation failed:")
        print(result.stderr[-2000:])

In [ ]:
import importlib
from pathlib import Path

import ipywidgets as widgets
import requests
import threddsclient
from IPython.display import display

print("Python:", sys.executable)
print("ipywidgets:", widgets.__version__)

# These two are only informative and are not always present outside JupyterLab
for _name in ("notebook", "jupyterlab_widgets"):
    try:
        print(f"{_name}:", importlib.import_module(_name).__version__)
    except Exception:
        print(f"{_name}: not available")

#### Helper modules

The helper scripts are expected in a `nrt_src/` folder next to this notebook. If the folder isn't
found, the cell says so explicitly instead of failing later with a confusing `ModuleNotFoundError`.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
SRC_DIR = NOTEBOOK_DIR / "nrt_src"

if not SRC_DIR.is_dir():
    raise FileNotFoundError(
        f"Helper folder not found: {SRC_DIR}\n"
        f"This notebook must sit next to the 'nrt_src' folder containing nrt_config.py, "
        f"nrt_utils.py, nrt_ui.py and actris_plotting_utilities.py."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    print(f"Added {SRC_DIR} to the Python path.")

import nrt_config
import nrt_utils as nu
import nrt_ui as nui
import actris_plotting_utilities as ap

if DEV_RELOAD:
    for _mod in (nrt_config, nu, ap, nui):
        importlib.reload(_mod)
    print("Helper modules reloaded (DEV_RELOAD is True).")

print(f"Loaded NRT helpers from: {SRC_DIR}")

## 2. Access and open datasets

### 2.1 THREDDS Data Catalog

ACTRIS In-Situ NRT data can accessed either by using the EBAS data portal (https://ebas-data.nilu.no/, Framework: "ACTRIS_NRT") or by a THREDDS Data Server (TDS), which provides catalog, metadata, and data access services for scientific data. The catalog is found here: https://thredds.nilu.no/thredds/catalog/actris_nrt/catalog.html.

For this respective example you will use the Opendap protocol (streaming) to access the data. The two catalogs are crawled over the network, so this step can take from a few seconds to a couple of minutes. Failures are reported with a clear message rather than a raw traceback.

In [ ]:
import time

def fetch_opendap_urls(catalog_url, label):
    """Crawl a THREDDS catalog and return its OPeNDAP URLs, reporting time taken and failures."""
    t0 = time.time()
    try:
        urls = threddsclient.opendap_urls(catalog_url)
    except Exception as exc:
        print(f"ERROR: could not read the {label} catalog ({type(exc).__name__}: {exc})")
        print(f"       Check your network connection and that {catalog_url} is reachable.")
        return []
    print(f"Found {len(urls)} OPeNDAP dataset URLs in the {label} catalog "
          f"({time.time() - t0:.1f} s).")
    return urls

actris_opendap_urls = fetch_opendap_urls(THREDDS_ACTRIS_NRT, "actris_nrt")
ebas_opendap_urls = fetch_opendap_urls(THREDDS_EBAS, "ebas")

In [ ]:
# The actris_nrt catalog should already contain NRT files, while the ebas
# catalog must be filtered because it also contains non-NRT files.
actris_nrt_opendap_urls = list(actris_opendap_urls)
ebas_nrt_opendap_urls = [u for u in ebas_opendap_urls if nu.is_nrt_file(u)]

print(f"Found {len(actris_nrt_opendap_urls)} NRT dataset URLs in actris_nrt catalog.")
print(f"Found {len(ebas_nrt_opendap_urls)} NRT dataset URLs in ebas catalog.")

if not actris_nrt_opendap_urls and not ebas_nrt_opendap_urls:
    raise RuntimeError(
        "No NRT datasets were found in either catalog. If the cell above reported an error, "
        "the THREDDS server was unreachable; otherwise the catalogs may be temporarily empty."
    )

### 2.2 Build the NRT file index

Set `SHOW_TABLES = True` in the Configuration cell if you want a printed table overview of files and
instruments.

In [ ]:
df_files, df_files_all, duplicate_files, pivot, summary = nu.build_combined_file_index(
    actris_nrt_opendap_urls,
    ebas_nrt_opendap_urls,
    show_tables=SHOW_TABLES,
)

print(f"Index built: {len(df_files)} unique NRT files "
      f"({len(df_files_all)} before de-duplication, {len(duplicate_files)} duplicates removed).")

if len(df_files) == 0:
    raise RuntimeError("The combined NRT file index is empty - there is nothing to plot. "
                        "Re-run the catalog access cells above and check for errors.")

## 3. Interactive search and plotting

Run the code block below once to launch the interactive ACTRIS-style NRT interface.

The interface uses the combined `df_files` index as the NRT equivalent of the ACTRIS metadata archive. Unlike the yearly ACTRIS In-Situ datasets, the NRT search remains filename-driven from THREDDS catalogs.
Anyhow, the search workflow follows the same logic as the ACTRIS In-Situ notebook:

```text
Instrument → Variable → Facility → File → Plot
```

Users can then select a custom time interval and interactively explore the selected NRT dataset using Plotly visualizations.

If the widgets ever fail to render (typically after a kernel restart, or when reopening a notebook
whose outputs were saved), re-run this cell: it closes any orphan widgets left over from a previous
session before rebuilding the interface.

The interface is tall — dropdowns, map and plots — so JupyterLab may put it in a scrolling box.
This cell is saved with output scrolling switched off, but if you still get a scrollbar,
right-click the output and choose *Disable Scrolling for Outputs*. The same menu has *Create New
View for Cell Output*, which opens the interface in a panel of its own that you can place beside
the notebook and resize freely — the most comfortable way to work with it on a small screen.

The interface opens on a worked example — absorption coefficient from the filter absorption
photometer at Sonnblick (AT0034G), 7 to 10 June 2025 — so that the first plot shows something
meaningful. Every selection is yours to change, and the defaults are ignored when the data do not
contain them. They are set in `DEFAULT_SELECTION` at the top of `nrt_src/nrt_ui.py`; passing
`nui.build_app(df_files, defaults={})` opens on the first available option instead.

In [ ]:
# Close orphan or disconnected widgets from a previous session to avoid "No such comm" errors.
# This also clears the widgets created by earlier cells in this notebook, which is harmless here
# because none of them are needed once the app is running.
widgets.Widget.close_all()

app = nui.build_app(df_files)
display(app)

<a id="export"></a>
## Keeping the results

This notebook **keeps nothing between runs**: the data are read from the THREDDS server as they are
plotted, and the figures are interactive Plotly charts that live only in this page. There is no
output folder to empty and no export cell.

**To keep a figure**, hover over it and use the **camera icon** in the toolbar that appears in the
top-right corner: it saves a PNG to your browser's download folder. The same toolbar offers zoom,
pan and autoscale, worth using on a long time series.

**To keep the data instead**, run the cell below after plotting. It writes the dataset currently
loaded in the interface — the selected variables, over the time window you chose — as a NetCDF file
next to this notebook, which you can then download from the file browser on the left (right-click
the file, *Download*).

In [ ]:
from pathlib import Path

ds = nui.current_dataset()

if ds is None:
    print("Nothing loaded yet: choose a file in the interface above and press Plot first.")
else:
    nc_path = Path("nrt_selection.nc")
    ds.to_netcdf(nc_path)
    print(f"Saved to {nc_path.resolve()}")
    print(f"Variables: {', '.join(ds.data_vars)}")
    print("Download it from the file browser on the left: right-click the file, then Download.")